# LLM Evaluation & Monitoring Dashboard

A notebook starter for evaluating LLM traces across answer-quality proxies, latency, estimated cost, success rate, feedback, and trends. Data is synthetic; this notebook does not call an LLM API or provide live monitoring. Token overlap is only an illustrative heuristic, not a definitive correctness score.

In [ ]:
%pip install -q pandas numpy matplotlib

In [ ]:
from pathlib import Path
import re, json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('llm_monitoring_outputs')
OUT.mkdir(exist_ok=True)

## 1. Sample LLM trace data
Replace these synthetic records with redacted traces from your application.

In [ ]:
records = [
 {"request_id":"r1","model":"model-a","task":"policy_qa","reference":"Refunds are available within 30 days.","answer":"Refunds are available within 30 days.","latency_ms":820,"input_tokens":120,"output_tokens":32,"cost_usd":0.0008,"status":"success","feedback":5},
 {"request_id":"r2","model":"model-b","task":"policy_qa","reference":"Use the password reset link on the sign-in page.","answer":"Use the password reset link on the sign-in page.","latency_ms":610,"input_tokens":110,"output_tokens":19,"cost_usd":0.0004,"status":"success","feedback":5},
 {"request_id":"r3","model":"model-a","task":"policy_qa","reference":"Refunds are available within 30 days.","answer":"Refunds are available for 90 days without restrictions.","latency_ms":730,"input_tokens":140,"output_tokens":22,"cost_usd":0.0005,"status":"success","feedback":1},
 {"request_id":"r4","model":"model-b","task":"summary","reference":"The release adds exports and fixes login issues.","answer":"The release adds exports and fixes login issues.","latency_ms":550,"input_tokens":260,"output_tokens":17,"cost_usd":0.0003,"status":"success","feedback":5},
 {"request_id":"r5","model":"model-a","task":"policy_qa","reference":"Update email from profile settings and verify it.","answer":"I cannot retrieve the policy right now. Please try later.","latency_ms":2400,"input_tokens":125,"output_tokens":18,"cost_usd":0.0006,"status":"error","feedback":2},
 {"request_id":"r6","model":"model-b","task":"summary","reference":"Enable MFA and rotate exposed credentials.","answer":"Enable MFA and rotate exposed credentials.","latency_ms":670,"input_tokens":280,"output_tokens":16,"cost_usd":0.0004,"status":"success","feedback":5},
 {"request_id":"r7","model":"model-a","task":"policy_qa","reference":"Billing details are in account settings.","answer":"Billing might be on a third-party website.","latency_ms":1800,"input_tokens":118,"output_tokens":24,"cost_usd":0.0006,"status":"success","feedback":2},
 {"request_id":"r8","model":"model-b","task":"summary","reference":"Maintenance starts at midnight and may last one hour.","answer":"Maintenance starts at midnight and may last about one hour.","latency_ms":890,"input_tokens":240,"output_tokens":20,"cost_usd":0.0007,"status":"success","feedback":4}
]
df=pd.DataFrame(records)
df['timestamp']=pd.date_range('2026-10-01 09:00',periods=len(df),freq='15min')
df

## 2. Calculate a simple reference-overlap proxy
Token overlap is easy to inspect, but it cannot reliably judge semantic equivalence or factual correctness.

In [ ]:
def tokens(text):
    return re.findall(r'\\b[a-z0-9]+\\b', str(text).lower())

def overlap(reference, answer):
    a, b = Counter(tokens(reference)), Counter(tokens(answer))
    overlap_n = sum((a & b).values())
    precision = overlap_n / sum(b.values()) if sum(b.values()) else 0
    recall = overlap_n / sum(a.values()) if sum(a.values()) else 0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0
    return pd.Series({'token_precision':precision,'token_recall':recall,'token_f1_proxy':f1})

df=pd.concat([df,df.apply(lambda r: overlap(r.reference,r.answer),axis=1)],axis=1)
df['is_success']=df.status.eq('success')
df['low_quality_proxy']=df.token_f1_proxy < 0.35
df[['request_id','model','status','token_f1_proxy','latency_ms','cost_usd']]

## 3. Headline KPIs

In [ ]:
kpis={
 'total_requests':int(len(df)),
 'success_rate_pct':round(100*df.is_success.mean(),2),
 'error_rate_pct':round(100*(~df.is_success).mean(),2),
 'average_latency_ms':round(df.latency_ms.mean(),2),
 'p95_latency_ms':round(df.latency_ms.quantile(.95),2),
 'total_estimated_cost_usd':round(df.cost_usd.sum(),6),
 'average_token_f1_proxy':round(df.token_f1_proxy.mean(),3),
 'average_feedback':round(df.feedback.mean(),2),
 'low_quality_proxy_rate_pct':round(100*df.low_quality_proxy.mean(),2)
}
pd.Series(kpis,name='value').to_frame()

## 4. Compare models

In [ ]:
model_summary=df.groupby('model').agg(
 requests=('request_id','count'),
 success_rate=('is_success','mean'),
 avg_latency_ms=('latency_ms','mean'),
 p95_latency_ms=('latency_ms',lambda s:s.quantile(.95)),
 total_cost_usd=('cost_usd','sum'),
 avg_token_f1_proxy=('token_f1_proxy','mean'),
 avg_feedback=('feedback','mean')
).reset_index()
model_summary['success_rate_pct']=(100*model_summary.pop('success_rate')).round(2)
model_summary.round(4)

## 5. Visualize latency, cost, quality, and success rate

In [ ]:
fig,ax=plt.subplots()
df.groupby('model').latency_ms.mean().plot(kind='bar',ax=ax)
ax.set(title='Average Latency by Model',ylabel='Milliseconds',xlabel='Model')
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots()
df.groupby('model').cost_usd.sum().plot(kind='bar',ax=ax)
ax.set(title='Total Estimated Cost by Model',ylabel='USD',xlabel='Model')
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots()
df.groupby('model').token_f1_proxy.mean().plot(kind='bar',ax=ax)
ax.set(title='Average Token-F1 Proxy by Model',ylabel='Token-F1 proxy (not correctness)',xlabel='Model')
plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots()
(100*df.groupby('model').is_success.mean()).plot(kind='bar',ax=ax)
ax.set(title='Success Rate by Model',ylabel='Percent',xlabel='Model',ylim=(0,100))
plt.tight_layout(); plt.show()

## 6. Review queue
Low token overlap, low user feedback, and errors are triage signals—not automatic proof of bad answers.

In [ ]:
review_queue=df.loc[(~df.is_success)|df.low_quality_proxy|(df.feedback<=2),
 ['request_id','model','task','reference','answer','status','token_f1_proxy','feedback']]
review_queue

## 7. Export dashboard data

In [ ]:
df.to_csv(OUT/'evaluation_records.csv',index=False)
model_summary.to_csv(OUT/'model_summary.csv',index=False)
pd.Series(kpis,name='value').to_csv(OUT/'dashboard_kpis.csv')
with open(OUT/'dashboard_kpis.json','w',encoding='utf-8') as f:
    json.dump(kpis,f,indent=2)
review_queue.to_csv(OUT/'review_queue.csv',index=False)
print('Exported:',*[p.name for p in sorted(OUT.iterdir())],sep='\\n- ')

## 8. Integrating real traces and production roadmap

Suggested fields: request ID, timestamp, model, task, redacted prompt/answer, reference or evaluator result, latency, token counts, cost, status, and feedback. Prefer provider-reported token usage and cost when available; otherwise calculate estimates from a versioned pricing configuration.

Next steps:
1. Add human-labelled datasets and task-specific rubrics.
2. Add calibrated semantic evaluation or an LLM-as-judge workflow.
3. Capture traces using an observability platform or OpenTelemetry.
4. Track prompt versions, retrieval quality, tool-call success, and safety.
5. Add alerts for latency, errors, cost spikes, and quality regressions.
6. Build a Streamlit dashboard or connect exports to a BI tool.
7. Add privacy controls, retention policies, and access control.

**Caution:** sample data and costs are illustrative. This notebook performs batch analysis, not real-time monitoring.